# Issue #1: ACN-Data API Connection

**Goal:** connect to the ACN-Data Caltech sessions endpoint with an API token, confirm the response structure and identify the session fields.

The token is entered with `getpass`, so it is never printed or saved in the notebook.

## Issue #1: API connection

In [1]:
import json
import time
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests

URL = "https://ev.caltech.edu/api/v1/sessions/caltech"
API_TOKEN = getpass("Enter your ACN API token: ")      # never printed or saved


def api_get(params, attempts=6, timeout=90):
    """GET with retries and back-off (5, 10, 20, 40, 60, 60 s). Raises after the last attempt."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(URL, params=params, auth=(API_TOKEN, ""), timeout=timeout)
            if r.status_code == 200:
                return r.json()
            last = f"HTTP {r.status_code}"
        except requests.exceptions.RequestException as e:
            last = type(e).__name__
        print(f"    {last} (attempt {attempt}/{attempts})")
        if attempt < attempts:
            time.sleep(min(60, 5 * 2 ** (attempt - 1)))
    raise RuntimeError(f"Request failed after {attempts} attempts ({last}). Re-run this cell to resume.")


first = api_get({"max_results": 25})
API_TOTAL = first["_meta"]["total"]
print("ACN API connection successful.")
print("Response keys:", list(first.keys()))
print("Fields per session:", list(first["_items"][0].keys()))
print("Sessions reported by the API:", f"{API_TOTAL:,}")

Enter your ACN API token:  ········


ACN API connection successful.
Response keys: ['_items', '_links', '_meta']
Fields per session: ['_id', 'userInputs', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'kWhDelivered', 'doneChargingTime', 'timezone', 'userID']
Sessions reported by the API: 31,424


**API fields:** `_id` (record id), `sessionID`, `stationID`, `spaceID`, `siteID`, `clusterID`, `connectionTime`, `disconnectTime`, `kWhDelivered`, `doneChargingTime`, `timezone`, `userID` (when available), `userInputs` (when available).

## Issue #1: API Connection Summary

- **Data source:** ACN-Data (Caltech site), REST API
- **Endpoint:** `https://ev.caltech.edu/api/v1/sessions/caltech`
- **Authentication:** API token (entered with `getpass`, not stored)
- **Response structure:** `_items` (sessions), `_links` (pagination), `_meta` (page, page size, total)
- **Fields per session:** 13 (listed above); total sessions reported by the API printed above
- **Collection:** implemented in Issue #2 (weekly time windows, resumable)